# P25 — Explorar los límites del aprendizaje por transferencia con un Transformer unificado texto a texto

## 1. Título y paper

**Paper:** *Exploring the Limits of Transfer Learning with a Unified Text-to-Text Transformer*  
**Autoría:** Colin Raffel, Noam Shazeer, Adam Roberts, Katherine Lee, Sharan Narang, Michael Matena, Yanqi Zhou, Wei Li, Peter J. Liu  
**Año y venue:** 2019 · arXiv:1910.10683 · JMLR 21(140), 2020  
**Nivel:** L3 · **Motor:** `t5`  
**Ficha completa:** [`P25_t5`](../../papers/foundational/P25_t5/README.md)

**Hito:** Todo problema de texto se reescribe como texto → texto: un solo modelo, una sola pérdida, cero cabezas específicas.

- [arXiv:1910.10683](https://arxiv.org/abs/1910.10683)
- [JMLR 21(140)](https://jmlr.org/papers/v21/20-074.html)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: Cada tarea exigía su propia cabeza —clasificación, regresión, extracción, generación— lo que impedía comparar objetivos, arquitecturas y datos en igualdad de condiciones.
2. Ejecutar una implementación mínima de la propuesta: Un marco unificado texto a texto, un estudio sistemático de todas las decisiones de diseño del preentrenamiento, y el corpus C4 (Colossal Clean Crawled Corpus).
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P08
- P09
- P24


## 4. Intuición

Cinco tareas, cinco arquitecturas, cinco formatos, cinco métricas. T5 pregunta: ¿y si todas fueran «te doy un texto, devuélveme un texto»? Entonces solo hay un modelo y una pérdida.


## 5. Concepto mínimo

```text
Antes:
    clasificar → cabeza con 2 logits + entropía cruzada
    regresión  → cabeza lineal + error cuadrático
    extracción → dos cabezas (inicio, fin) + entropía cruzada
    generación → decoder + verosimilitud

T5:
    TODO      → texto de entrada con prefijo → texto de salida
    pérdida   → maximizar log p(texto_salida | texto_entrada)
```


## 6. Código explicado

El motor muestra cinco tareas reescritas al mismo formato.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('t5', seed=7)['result']
for t in r['tareas']:
    print(f"{t['tarea']:<26} antes: {t['clasico']}")
    print(f"{'':<26} in : {t['entrada'][:62]}")
    print(f"{'':<26} out: {t['salida']}\n")

## 7. Predicción antes de ejecutar

1. ¿Cómo se emite una regresión (un número real) como texto?
2. ¿Qué se pierde al hacerlo?
3. Si todas las tareas comparten pérdida, ¿qué distingue una de otra?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
r = run_paper_lab('t5', seed=7)['result']
print('cabezas específicas antes :', r['cabezas_especificas_antes'])
print('cabezas específicas después:', r['cabezas_especificas_despues'])
print('objetivo único            :', r['objetivo_unico'])
print('lo único que cambia       :', r['que_cambia_por_tarea'])

## 9. Salida interpretable

Cinco tipos de cabeza distintos se reducen a cero. Lo único que distingue una tarea de otra es el **prefijo del texto de entrada**. Eso es lo que permite el estudio sistemático del paper: se pueden comparar objetivos, arquitecturas y corpus sin que la métrica cambie de significado.


## 10. Comentario pedagógico

La contribución más citada es el marco, pero la más valiosa es el **estudio**: decenas de experimentos controlados comparando objetivos de preentrenamiento, arquitecturas y tamaños de corpus. Es un paper de ingeniería empírica rigurosa, no una idea suelta.


## 11. Error o anti-patrón deliberado

Anti-patrón: emitir números como texto sin pensar en la precisión.


In [ ]:
for real in (4.2, 4.25, 0.333333, 12345.678):
    print(f'valor {real:<12} → texto "{real:.1f}"  (se pierde todo lo que sigue)')

## 12. Corrección

Por eso el paper discretiza la escala de la tarea de similitud a incrementos fijos:


In [ ]:
def discretizar(x, paso=0.2):
    return round(round(x / paso) * paso, 1)
for real in (4.2, 4.25, 4.31, 4.9):
    print(f'{real} → {discretizar(real)}  (el modelo solo tiene que acertar una de 26 clases)')

## 13. Desafío guiado

Reescribe una tarea propia al formato texto → texto y define su prefijo.


In [ ]:
mi_tarea = {'prefijo': 'detectar sentimiento: ',
            'entrada': 'detectar sentimiento: el envío llegó tarde y roto',
            'salida': 'negativo'}
show(mi_tarea)
print('¿qué cabeza específica necesita este modelo? ninguna')

## 14. Desafío autónomo

Toma tres tareas de un benchmark público, reescríbelas al formato texto → texto y ajusta un modelo pequeño de encoder-decoder abierto. Compara con entrenar tres modelos con cabezas específicas: reporta exactitud, parámetros totales y tiempo.


## 15. Evidencia de aprendizaje

Guarda las cinco tareas reescritas, la cuenta de cabezas antes y después, y tu explicación del coste de precisión al emitir números como texto.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P25_t5/README.md) · evaluación formal: [`assessments/papers/P25_t5.md`](../../assessments/papers/P25_t5.md)


## 16. Cierre

Un formato único para todas las tareas de texto. La siguiente pregunta ya no es de formato sino de **decisión**: qué hacer, en qué orden, y cómo saber si salió bien.


## 17. Conexión con el siguiente hito

- P11
- P22

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
